# 在 Google Colab 上训练 Arabic Moonshine LoRA

这个 notebook 用于运行 `UsefulSensors/moonshine-base-ar` 的阿拉伯语 LoRA 微调流程。

请在 Colab 中使用 GPU 运行时：**运行时 > 更改运行时类型 > GPU**。数据集压缩包会直接从 OSS 下载到 Colab 本地磁盘，Google Drive 只用于保存日志和最终训练产物。


## 1. 检查运行时

确认 Colab 已经分配到 GPU。CPU 也能机械地跑通流程，但训练会非常慢。


In [ ]:
!nvidia-smi || true


## 2. 配置路径

如果你的 GitHub 仓库、分支、OSS 下载链接或 Google Drive 输出目录不同，请先修改下面这些变量。


In [ ]:
REPO_URL = "https://github.com/MarshalXu/finetune-moonshine-asr.git"
BRANCH = "shawnx/peft_ft"
WORKDIR = "/content/finetune-moonshine-asr"

# Data source: "oss" is recommended for large datasets. Use "drive" as a fallback.
DATA_SOURCE = "oss"
OSS_ZIP_URL = ""  # Paste your public or signed OSS zip URL here.

DRIVE_ROOT = "/content/drive/MyDrive/moonshine"
DRIVE_ZIP = f"{DRIVE_ROOT}/whisper_ar_manifist0508.zip"
LOCAL_ZIP = "/content/downloads/whisper_ar_manifist0508.zip"

DATASET_NAME = "whisper_ar_manifist0508"
RAW_DATA_DIR = f"{WORKDIR}/datasets/{DATASET_NAME}"
HF_DATASET_DIR = f"{WORKDIR}/datasets/{DATASET_NAME}_hf"
OUTPUT_DIR = f"{WORKDIR}/results-moonshine-base-ar-lora-colab"
DRIVE_OUTPUT_DIR = f"{DRIVE_ROOT}/results-moonshine-base-ar-lora-colab"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Data source:", DATA_SOURCE)
print("OSS URL set:", bool(OSS_ZIP_URL))
print("Local zip:", LOCAL_ZIP)
print("Drive output:", DRIVE_OUTPUT_DIR)


## 3. 克隆项目

拉取包含阿拉伯语 LoRA 微调流程和 Colab notebook 的分支。


In [ ]:
import os
import subprocess


def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

run(f"rm -rf {WORKDIR}")
run(f"git clone --branch {BRANCH} {REPO_URL} {WORKDIR}")
os.chdir(WORKDIR)
print("cwd:", os.getcwd())


## 4. 安装依赖

Colab 已经预装 PyTorch。这里安装项目依赖，包括 `peft`。当前 Colab 数据读取路径使用 `soundfile`，不依赖 TorchCodec。


In [ ]:
!python -m pip install -q -r requirements.txt

import torch
import transformers
import datasets
import peft

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("peft", peft.__version__)


## 5. 挂载 Google Drive

Google Drive 用于保存训练日志和最终 adapter。数据集压缩包默认从 `OSS_ZIP_URL` 下载，不需要先上传到 Drive。


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!mkdir -p "$DRIVE_ROOT"


## 6. 准备原始数据集

这一步会从 OSS 下载数据集压缩包，或者在需要时使用 Drive 里的压缩包作为备用来源，然后解压到项目的 `datasets/` 目录。如果原始数据集已经解压在目标目录，这个单元会直接跳过。


In [ ]:
from pathlib import Path
import shlex

raw_dir = Path(RAW_DATA_DIR)
local_zip = Path(LOCAL_ZIP)
drive_zip = Path(DRIVE_ZIP)

if raw_dir.exists():
    print(f"Raw dataset already exists: {raw_dir}")
else:
    raw_dir.parent.mkdir(parents=True, exist_ok=True)
    local_zip.parent.mkdir(parents=True, exist_ok=True)

    if DATA_SOURCE == "oss":
        if not OSS_ZIP_URL:
            raise ValueError("Set OSS_ZIP_URL in the configuration cell before running this cell.")
        print(f"Downloading dataset from OSS -> {local_zip}")
        quoted_url = shlex.quote(OSS_ZIP_URL)
        quoted_out = shlex.quote(str(local_zip))
        run(f"curl -L --retry 5 --retry-delay 5 --connect-timeout 30 -o {quoted_out} {quoted_url}")
    elif DATA_SOURCE == "drive":
        if not drive_zip.exists():
            raise FileNotFoundError(f"Drive zip not found: {drive_zip}")
        print(f"Copying Drive zip -> {local_zip}")
        run(f"cp {shlex.quote(str(drive_zip))} {shlex.quote(str(local_zip))}")
    else:
        raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

    print(f"Unzipping {local_zip} -> {raw_dir.parent}")
    run(f"unzip -q -o {shlex.quote(str(local_zip))} -d {shlex.quote(str(raw_dir.parent))}")
    if not raw_dir.exists():
        raise FileNotFoundError(f"Expected extracted dataset at {raw_dir}. Check zip top-level folder name.")

print("train manifest:", raw_dir / "train.jsonl", (raw_dir / "train.jsonl").exists())
print("test manifest:", raw_dir / "test.jsonl", (raw_dir / "test.jsonl").exists())


## 7. 转换并校验数据集

这一步会生成 Hugging Face `DatasetDict`，并跳过 `soundfile` 无法解码的音频文件。由于会逐条校验音频，数据量较大时可能需要几分钟。


In [ ]:
!python scripts/prepare_manifest_dataset.py \
  --manifest-dir "$RAW_DATA_DIR" \
  --output "$HF_DATASET_DIR" \
  --validate-audio \
  --validation-backend soundfile \
  --no-cast-audio \
  --skip-invalid-audio \
  --overwrite


## 8. 创建 Colab 训练配置

这一步会复制仓库里的 LoRA 配置，并覆盖为 Colab 的输出路径。默认训练 6 个 epoch，每个 epoch 后评估和保存 checkpoint，并根据 WER 选择 best model。


In [ ]:
import yaml
from pathlib import Path

base_config_path = Path("configs/moonshine_base_ar_lora_cpu_train.yaml")
colab_config_path = Path("configs/moonshine_base_ar_lora_colab.yaml")
config = yaml.safe_load(base_config_path.read_text())
config["dataset"]["path"] = HF_DATASET_DIR
config["dataset"]["cast_audio"] = False
config["training"]["output_dir"] = OUTPUT_DIR
config["training"]["logging_dir"] = f"{WORKDIR}/logs/moonshine-base-ar-lora-colab"

# GPU-friendly baseline. Adjust if you hit memory limits.
config["training"]["per_device_train_batch_size"] = 4
config["training"]["per_device_eval_batch_size"] = 4
config["training"]["gradient_accumulation_steps"] = 4
config["training"]["fp16"] = True
config["training"]["fp16_full_eval"] = True

colab_config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print(colab_config_path.read_text())


## 9. 开始训练

训练日志会实时显示。每个 epoch 结束后会看到 `eval_wer` 和 `eval_cer`。训练结束时会按照 WER 自动加载最优 adapter。


In [ ]:
!python -u train.py --config configs/moonshine_base_ar_lora_colab.yaml 2>&1 | tee "$DRIVE_ROOT/lora_train.log"


## 10. 断点续训

如果 Colab 断开，可以使用这一节继续训练。把 checkpoint 路径改成输出目录里最新的 checkpoint。


In [ ]:
!find "$OUTPUT_DIR" -maxdepth 1 -type d -name 'checkpoint-*' | sort

# Example:
# !python -u train.py --config configs/moonshine_base_ar_lora_colab.yaml \
#   --resume "$OUTPUT_DIR/checkpoint-XXXX" 2>&1 | tee -a "$DRIVE_ROOT/lora_train_resume.log"


## 11. 复制产物到 Drive

最终产物是 LoRA adapter，不是完整模型。后续推理时需要同时加载 base model 和这个 adapter。


In [ ]:
!mkdir -p "$DRIVE_OUTPUT_DIR"
!rsync -av "$OUTPUT_DIR/" "$DRIVE_OUTPUT_DIR/"
!find "$DRIVE_OUTPUT_DIR" -maxdepth 2 -type f | sort | sed -n '1,80p'
